# LLM Cold-Start Benchmark - Colab driver

Phase-decomposed cold-start measurement for open-source LLM inference. Runs on a
free **T4** runtime (Runtime -> Change runtime type -> GPU).

Cold start is not one number - it's `pull -> import -> weight-load -> to-device ->
first-forward`, and which phase dominates depends on model size, the storage tier
the weights sit on, the checkpoint format, and the engine. This notebook
decomposes those phases, fits load-time vs model-size, and extrapolates to
production sizes.

## 1. Set up the repo
Upload `coldstart-lab.zip`, then unzip and install.

In [ ]:
from google.colab import files
up = files.upload()  # pick coldstart-lab.zip

In [ ]:
!unzip -o -q coldstart-lab.zip
%cd coldstart-lab
!pip install -q -e .[gpu]
# vllm/bitsandbytes are heavy. if you don't need the engine experiment:
# !pip install -q -e .[dev] accelerate bitsandbytes

## 2. Sanity checks

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
import torch
print("cuda:", torch.cuda.is_available(), "| torch:", torch.__version__)
from coldstart import models, experiments, report
from coldstart.runner import run_config
print("catalog size:", len(models.CATALOG))

## 3. Size-scaling sweep
Load each model cold (fresh subprocess per run, page cache dropped between runs),
decompose phases, fit `load_s ~ slope*GB + intercept`, extrapolate. fp16 on GPU.

In [ ]:
specs, results = {}, {}
for mid in models.DEFAULT_T4_SWEEP:
    specs[mid] = models.find(mid)
    cfg = {"backend": "transformers", "model_id": mid, "device": "cuda:0", "dtype": "float16"}
    print("running", mid, "...")
    results[mid] = run_config(cfg, repeats=3, drop_cache=True, warmup=True)

for mid, recs in results.items():
    s = report.summarize(recs); ph = s["phases"]
    print(f"{mid:42s} load p50={ph['load']['p50']:.2f}s  "
          f"ff p50={ph.get('first_forward',{}).get('p50',0):.2f}s  "
          f"total p50={s['total']['p50']:.2f}s")

In [ ]:
curve = report.curve_from_sweep(results, specs)
print(curve)
extrap = report.extrapolate(curve, {"13B": 26.0, "27B": 54.0, "70B": 140.0})
extrap

## 4. Phase breakdown plot

In [ ]:
import matplotlib.pyplot as plt, numpy as np
order = list(results.keys())
phase_names = ["import", "load", "tokenizer", "to_device", "first_forward"]
vals = {p: [report.summarize(results[m])["phases"].get(p, {}).get("p50", 0) for m in order]
        for p in phase_names}
fig, ax = plt.subplots(figsize=(10, 5)); bottom = np.zeros(len(order))
for p in phase_names:
    ax.bar([m.split("/")[-1] for m in order], vals[p], bottom=bottom, label=p)
    bottom += np.array(vals[p])
ax.set_ylabel("seconds (p50)"); ax.set_title("Cold-start phase breakdown"); ax.legend()
plt.xticks(rotation=30, ha="right"); plt.tight_layout(); plt.show()

In [ ]:
sizes = [specs[m].approx_gb for m in order]
loads = [report.summarize(results[m])["phases"]["load"]["p50"] for m in order]
xs = np.linspace(0, 60, 100)
plt.figure(figsize=(8,5)); plt.scatter(sizes, loads, label="measured")
plt.plot(xs, curve.slope_s_per_gb*xs + curve.intercept_s, "--",
         label=f"fit R2={curve.r2:.2f}, ~{curve.eff_bandwidth_MBps:.0f} MB/s")
for name, d in extrap.items():
    plt.scatter([d["gb"]],[d["predicted_load_s"]], marker="x", s=80)
    plt.annotate(name,(d["gb"],d["predicted_load_s"]))
plt.xlabel("model size (GB, fp16)"); plt.ylabel("weight-load p50 (s)")
plt.title("Load time scales with weight bytes"); plt.legend(); plt.show()

## 5. Storage tier: local NVMe vs network-attached (Drive)
Closest analog to a shared-filesystem setup. Drive is FUSE-backed, so its numbers
include the FUSE layer's caching - realistic, but not a clean page-cache drop.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
tiers = {"local_nvme": "/content/staging",
         "drive_nas": "/content/drive/MyDrive/coldstart_staging"}
tier_res = experiments.storage_tier("Qwen/Qwen2.5-1.5B-Instruct", tiers=tiers,
                                    repeats=3, device="cuda:0")
for tier, s in report.summarize_arms(tier_res).items():
    bw = tier_res[tier][0].meta.get("stage_MBps")
    print(f"{tier:12s} load p50={s['phases']['load']['p50']:.2f}s  stage ~{bw} MB/s")
report.diff_total(tier_res, baseline="drive_nas")

## 6. Quantization: fp16 vs 4-bit (7B)

In [ ]:
quant_res = experiments.quantization("Qwen/Qwen2.5-7B-Instruct", repeats=3)
report.summarize_arms(quant_res)

## 7. Engine cold start: vLLM with vs without CUDA-graph capture
Isolates how much of vLLM startup is graph capture - the phase snapshot/restore
(Modal, InferX) skips by restoring post-capture state.

In [ ]:
graph_res = experiments.engine_cuda_graphs("Qwen/Qwen2.5-0.5B-Instruct", repeats=2)
report.summarize_arms(graph_res)

## 8. Gated / production models (optional)

In [ ]:
# from huggingface_hub import login; login("hf_...")
# for mid in ["meta-llama/Llama-3.2-1B-Instruct", "meta-llama/Llama-3.2-3B-Instruct"]:
#     specs[mid] = models.find(mid)
#     cfg = {"backend":"transformers","model_id":mid,"device":"cuda:0","dtype":"float16"}
#     results[mid] = run_config(cfg, repeats=3, drop_cache=True, warmup=True)

## 9. Save results.json

In [ ]:
import json
payload = {
    "sweep": {m: report.summarize(r) for m, r in results.items()},
    "load_curve": {"slope_s_per_gb": curve.slope_s_per_gb, "intercept_s": curve.intercept_s,
                   "r2": curve.r2, "eff_bandwidth_MBps": curve.eff_bandwidth_MBps},
    "extrapolation": extrap,
    "storage_tier": report.summarize_arms(tier_res),
    "quantization": report.summarize_arms(quant_res),
}
with open("results.json","w") as f: json.dump(payload, f, indent=2, default=str)
from google.colab import files; files.download("results.json")